# 13 — Assembly101 Coarse Annotations to MS-TCN Format

This notebook converts the official **coarse Assembly101 annotations** into a reproducible MS-TCN-style dataset description.

The official Assembly101 annotation repository specifies that:

- coarse annotations are used for Temporal Action Segmentation;
- raw videos are 60 fps;
- annotation frame numbers refer to frames extracted at 30 fps;
- every coarse label row contains:
  `start_frame end_frame action_cls`;
- official split files are provided separately for assembly/disassembly and train/validation/test.

This notebook creates one TAS sequence for each:

```text
assembly_<recording_name>
disassembly_<recording_name>
```

Each sequence is represented as a temporal crop of the original recording:

```text
clip start = first annotated coarse frame
clip end   = last annotated coarse frame
```

The output does **not** download videos or extract visual features yet.

## Output

```text
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/
├── features/                         # empty placeholder for later features
├── groundTruth/                      # dense labels at annotation 30 fps
├── splits/
│   ├── train.split1.bundle
│   ├── val.split1.bundle
│   ├── test.split1.bundle
│   └── train_val.split1.bundle
├── mapping.txt
├── class_metadata.csv
├── segment_manifest.csv
├── sequence_manifest.csv
├── sequence_video_manifest_v1.csv
├── recording_download_manifest_v1.csv
├── procedurevrl_extraction_manifest_v1.csv
├── validation_report.csv
└── dataset_summary.json
```

The next notebook will use the recording/crop manifest to download the required `v1` videos and prepare ProcedureVRL extraction.

## 1. Mount Drive and imports

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import json
import re
import shutil
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

print("Imports OK")

Mounted at /content/drive
Imports OK


## 2. Configuration

In [2]:
DRIVE_ROOT = Path("/content/drive/MyDrive/mmf_tas_lab_data")
ASSEMBLY_ROOT = DRIVE_ROOT / "assembly101"

ANNOTATION_ROOT = ASSEMBLY_ROOT / "annotations"
COARSE_ROOT = ANNOTATION_ROOT / "coarse-annotations"
COARSE_LABEL_DIR = COARSE_ROOT / "coarse_labels"
COARSE_SPLIT_DIR = COARSE_ROOT / "coarse_splits"

ACTIONS_PATH = COARSE_ROOT / "actions.csv"
COARSE_SEQ_VIEWS_PATH = COARSE_ROOT / "coarse_seq_views.txt"
TAIL_ACTIONS_PATH = COARSE_ROOT / "tail_actions.txt"

RECORDING_INVENTORY_PATH = (
    ASSEMBLY_ROOT / "manifests" / "recording_file_inventory.csv"
)

OUT_ROOT = (
    DRIVE_ROOT
    / "text_assisted_tas"
    / "assembly101"
    / "coarse_mstcn_format"
)

GT_DIR = OUT_ROOT / "groundTruth"
FEATURE_DIR = OUT_ROOT / "features"
SPLIT_OUT_DIR = OUT_ROOT / "splits"

SPLIT_ID = 1

# We start with one fixed RGB view, consistent with notebook 12.
SELECTED_VIEW = "v1"
SELECTED_CAMERA_FILE = "C10095_rgb.mp4"

# Official annotations use frame indices at 30 fps.
ANNOTATION_FPS = 30.0
RAW_VIDEO_FPS_DESCRIPTION = "raw recordings are nominally 60 fps"

# End-frame convention:
#   "auto"      -> infer from adjacency statistics
#   "exclusive" -> [start, end)
#   "inclusive" -> [start, end]
END_FRAME_CONVENTION = "auto"

# Fill temporal gaps inside each assembly/disassembly crop with this class.
BACKGROUND_LABEL = "background"

# Fail on overlapping segments with conflicting labels.
FAIL_ON_CONFLICTING_OVERLAPS = True

# For quick debugging only. Keep None for the full conversion.
MAX_SEQUENCES = None

for path in [OUT_ROOT, GT_DIR, FEATURE_DIR, SPLIT_OUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("ASSEMBLY_ROOT:", ASSEMBLY_ROOT)
print("COARSE_ROOT:", COARSE_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("SELECTED_VIEW:", SELECTED_VIEW)
print("SELECTED_CAMERA_FILE:", SELECTED_CAMERA_FILE)

ASSEMBLY_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/assembly101
COARSE_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations/coarse-annotations
OUT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format
SELECTED_VIEW: v1
SELECTED_CAMERA_FILE: C10095_rgb.mp4


## 3. Validate required inputs

In [3]:
required_paths = {
    "Annotation root": ANNOTATION_ROOT,
    "Coarse annotation root": COARSE_ROOT,
    "Coarse labels": COARSE_LABEL_DIR,
    "Coarse splits": COARSE_SPLIT_DIR,
    "Actions CSV": ACTIONS_PATH,
    "Sequence/view list": COARSE_SEQ_VIEWS_PATH,
    "Recording inventory from notebook 12": RECORDING_INVENTORY_PATH,
}

for name, path in required_paths.items():
    print(f"{name}: {path} -> exists={path.exists()}")
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {name}: {path}")

label_files = sorted(COARSE_LABEL_DIR.glob("*.txt"))
split_files = sorted(COARSE_SPLIT_DIR.glob("*.txt"))

print("\nCoarse label files:", len(label_files))
print("Official split files:", len(split_files))

if not label_files:
    raise RuntimeError("No coarse label files were found.")

Annotation root: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations -> exists=True
Coarse annotation root: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations/coarse-annotations -> exists=True
Coarse labels: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations/coarse-annotations/coarse_labels -> exists=True
Coarse splits: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations/coarse-annotations/coarse_splits -> exists=True
Actions CSV: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations/coarse-annotations/actions.csv -> exists=True
Sequence/view list: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations/coarse-annotations/coarse_seq_views.txt -> exists=True
Recording inventory from notebook 12: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/manifests/recording_file_inventory.csv -> exists=True

Coarse label files: 680
Official split files: 6


## 4. Display the official annotation documentation

In [4]:
readme_candidates = [
    ANNOTATION_ROOT / "README.md",
    COARSE_ROOT / "README.md",
]

for path in readme_candidates:
    print("\n" + "=" * 100)
    print(path)
    print("=" * 100)

    if path.exists():
        text = path.read_text(encoding="utf-8", errors="replace")
        print(text[:20000])
    else:
        print("[not present in the Hugging Face snapshot]")


/content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations/README.md
# Assembly101 annotations

The annotations are divided into 2 granularities:
- `fine-grained-annotations`: used for Action Recognition and Action Anticipation benchmarks
- `coarse-annotations`: used for Temporal Action Segmentation benchmark

```
Raw videos are 60fps but the frame numbers in all the annotations are provided after extracting them from the raw video at 30fps.
```

If you use our dataset, kindly cite:
```
@article{sener2022assembly101,
    title = {Assembly101: A Large-Scale Multi-View Video Dataset for Understanding Procedural Activities},
    author = {F. Sener and D. Chatterjee and D. Shelepov and K. He and D. Singhania and R. Wang and A. Yao},
    journal = {CVPR 2022},
}
```

## Fine-Grained Annotations
We have 1380 fine-grained actions composed of a combination of 90 objects and 24 verbs.

The files present under the `fine-grained-annotations` folder are:
- `actions.csv`
- `train.csv`
- `vali

## 5. Load and validate the 202 coarse action classes

In [5]:
actions = pd.read_csv(ACTIONS_PATH)

expected_action_columns = {
    "action_id",
    "verb_id",
    "noun_id",
    "action_cls",
    "verb_cls",
    "noun_cls",
}

missing_action_columns = expected_action_columns - set(actions.columns)
if missing_action_columns:
    raise ValueError(
        f"actions.csv is missing columns: {sorted(missing_action_columns)}. "
        f"Found: {list(actions.columns)}"
    )

actions = actions.copy()
actions["action_id"] = pd.to_numeric(actions["action_id"], errors="raise").astype(int)
actions["action_cls"] = (
    actions["action_cls"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

if actions["action_id"].duplicated().any():
    raise ValueError("Duplicate action_id values in actions.csv")

if actions["action_cls"].duplicated().any():
    duplicate_names = actions.loc[
        actions["action_cls"].duplicated(keep=False),
        ["action_id", "action_cls"],
    ]
    display(duplicate_names)
    raise ValueError("Duplicate action_cls values in actions.csv")

actions = actions.sort_values("action_id").reset_index(drop=True)

print("Number of official coarse classes:", len(actions))
print("Action ID range:", actions["action_id"].min(), "to", actions["action_id"].max())
print("Contiguous official IDs:", actions["action_id"].tolist() == list(range(len(actions))))

display(actions.head(20))
display(actions.tail(10))

if len(actions) != 202:
    print(
        "WARNING: expected 202 official coarse classes, "
        f"but actions.csv contains {len(actions)}."
    )

Number of official coarse classes: 202
Action ID range: 0 to 201
Contiguous official IDs: True


,action_id,verb_id,noun_id,action_cls,verb_cls,noun_cls
0,0,2,4,inspect toy,inspect,toy
1,1,0,0,attach cabin,attach,cabin
2,2,1,0,detach cabin,detach,cabin
3,3,1,3,detach wheel,detach,wheel
4,4,0,3,attach wheel,attach,wheel
5,5,4,1,screw chassis,screw,chassis
6,6,6,9,demonstrate functionality,demonstrate,functionality
7,7,3,1,unscrew chassis,unscrew,chassis
8,8,0,2,attach interior,attach,interior
9,9,1,5,detach roof,detach,roof


,action_id,verb_id,noun_id,action_cls,verb_cls,noun_cls
192,192,5,48,attempt to attach rocker panel,attempt to attach,rocker panel
193,193,4,26,screw roller,screw,roller
194,194,3,29,unscrew engine cover,unscrew,engine cover
195,195,4,30,screw mixer stand,screw,mixer stand
196,196,4,20,screw grill,screw,grill
197,197,5,28,attempt to attach strap,attempt to attach,strap
198,198,0,59,attach fire equipment,attach,fire equipment
199,199,1,60,detach battery,detach,battery
200,200,0,60,attach battery,attach,battery
201,201,5,27,attempt to attach sound module,attempt to attach,sound module


## 6. Inspect representative coarse-label files

In [6]:
def nonempty_lines(path: Path):
    return [
        line.strip()
        for line in path.read_text(encoding="utf-8", errors="replace").splitlines()
        if line.strip()
    ]

sample_label_files = []

for prefix in ["assembly_", "disassembly_"]:
    matches = [p for p in label_files if p.stem.startswith(prefix)]
    if matches:
        sample_label_files.extend(matches[:2])

sample_label_files = list(dict.fromkeys(sample_label_files))

for path in sample_label_files:
    print("\n" + "=" * 100)
    print(path.name)
    print("=" * 100)
    for line in nonempty_lines(path)[:20]:
        print(repr(line))


assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt
'000004457\t000004560\tattach chassis'
'000004560\t000004792\tscrew chassis'
'000004792\t000005163\tattach body'
'000005163\t000005248\tattach cabin'
'000005248\t000005642\tscrew chassis'
'000005642\t000005809\tposition figurine'
'000005809\t000006522\tattach bucket'
'000006522\t000007065\tattach excavator arm'
'000007065\t000008070\tattach track'

assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt
'000002833\t000003156\tattach interior'
'000003156\t000004607\tattach wheel'
'000004607\t000005151\tattach turntable top'
'000005151\t000005613\tattach hook'
'000005613\t000006110\tattach crane arm'
'000006110\t000006394\tattach cabin'
'000006394\t000006959\tdemonstrate functionality'

disassembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt
'000000188\t000001609\tdetach track'
'000001609\t000002011\tdetach excavator arm'
'000002011\t000002407\tdetach bucket'
'00000240

## 7. Parse all coarse temporal segments

In [7]:
def normalize_label_text(value):
    return re.sub(r"\s+", " ", str(value).strip())


def parse_activity_and_recording(sequence_stem: str):
    if sequence_stem.startswith("assembly_"):
        return "assembly", sequence_stem[len("assembly_"):]
    if sequence_stem.startswith("disassembly_"):
        return "disassembly", sequence_stem[len("disassembly_"):]
    raise ValueError(
        f"Unexpected sequence name {sequence_stem!r}; "
        "expected assembly_... or disassembly_..."
    )


def parse_coarse_label_file(path: Path):
    rows = []

    for line_number, line in enumerate(nonempty_lines(path), start=1):
        parts = line.split(maxsplit=2)

        if len(parts) != 3:
            raise ValueError(
                f"Expected three fields in {path.name}:{line_number}, got: {line!r}"
            )

        start_raw, end_raw, action_cls_raw = parts

        try:
            start_frame = int(start_raw)
            end_frame = int(end_raw)
        except ValueError as e:
            raise ValueError(
                f"Invalid frame indices in {path.name}:{line_number}: {line!r}"
            ) from e

        action_cls = normalize_label_text(action_cls_raw)

        if start_frame < 0 or end_frame < 0:
            raise ValueError(f"Negative frame index in {path.name}:{line_number}")

        if end_frame < start_frame:
            raise ValueError(
                f"end_frame < start_frame in {path.name}:{line_number}: {line!r}"
            )

        rows.append({
            "sequence_id": path.stem,
            "annotation_file": path.name,
            "line_number": line_number,
            "start_frame_30fps_raw": start_frame,
            "end_frame_30fps_raw": end_frame,
            "action_cls": action_cls,
        })

    if not rows:
        raise ValueError(f"Empty coarse annotation file: {path}")

    activity, recording_name = parse_activity_and_recording(path.stem)

    for row in rows:
        row["activity"] = activity
        row["recording_name"] = recording_name

    return rows


files_to_parse = label_files
if MAX_SEQUENCES is not None:
    files_to_parse = files_to_parse[:MAX_SEQUENCES]

all_segment_rows = []

for path in files_to_parse:
    all_segment_rows.extend(parse_coarse_label_file(path))

segments_raw = pd.DataFrame(all_segment_rows)
segments_raw = segments_raw.sort_values(
    ["sequence_id", "start_frame_30fps_raw", "end_frame_30fps_raw"]
).reset_index(drop=True)

print("Parsed sequence files:", segments_raw["sequence_id"].nunique())
print("Parsed segments:", len(segments_raw))
print("Assembly sequences:", (segments_raw["activity"] == "assembly").groupby(segments_raw["sequence_id"]).first().sum())
print("Disassembly sequences:", (segments_raw["activity"] == "disassembly").groupby(segments_raw["sequence_id"]).first().sum())
print("Unique labels used:", segments_raw["action_cls"].nunique())

display(segments_raw.head(30))

Parsed sequence files: 680
Parsed segments: 8753
Assembly sequences: 334
Disassembly sequences: 346
Unique labels used: 202


,sequence_id,annotation_file,line_number,start_frame_30fps_raw,end_frame_30fps_raw,action_cls,activity,recording_name
0,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,1,4457,4560,attach chassis,assembly,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724
1,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,2,4560,4792,screw chassis,assembly,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724
2,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,3,4792,5163,attach body,assembly,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724
3,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,4,5163,5248,attach cabin,assembly,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724
4,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,5,5248,5642,screw chassis,assembly,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724
5,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,6,5642,5809,position figurine,assembly,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724
6,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,7,5809,6522,attach bucket,assembly,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724
7,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,8,6522,7065,attach excavator arm,assembly,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724
8,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,9,7065,8070,attach track,assembly,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724
9,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt,1,2833,3156,attach interior,assembly,nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253


## 8. Validate labels against `actions.csv`

In [8]:
official_label_set = set(actions["action_cls"])
used_label_set = set(segments_raw["action_cls"])

unknown_labels = sorted(used_label_set - official_label_set)
unused_official_labels = sorted(official_label_set - used_label_set)

print("Unknown labels in segment files:", len(unknown_labels))
print("Official labels unused in downloaded segments:", len(unused_official_labels))

if unknown_labels:
    print("\nUnknown label examples:")
    for label in unknown_labels[:100]:
        print(repr(label))
    raise ValueError(
        "Some coarse segment labels are absent from actions.csv. "
        "Do not continue until label normalization is resolved."
    )

label_to_official_id = dict(
    zip(actions["action_cls"], actions["action_id"])
)

segments_raw["official_action_id"] = (
    segments_raw["action_cls"]
    .map(label_to_official_id)
    .astype(int)
)

display(
    segments_raw[
        ["action_cls", "official_action_id"]
    ]
    .drop_duplicates()
    .sort_values("official_action_id")
    .head(30)
)

Unknown labels in segment files: 0
Official labels unused in downloaded segments: 0


,action_cls,official_action_id
24,inspect toy,0
3,attach cabin,1
61,detach cabin,2
2293,detach wheel,3
10,attach wheel,4
1,screw chassis,5
15,demonstrate functionality,6
157,unscrew chassis,7
9,attach interior,8
128,detach roof,9


## 9. Infer whether annotation end frames are inclusive or exclusive

In [9]:
adjacency_rows = []

for sequence_id, group in segments_raw.groupby("sequence_id", sort=False):
    group = group.sort_values(
        ["start_frame_30fps_raw", "end_frame_30fps_raw"]
    ).reset_index(drop=True)

    for i in range(len(group) - 1):
        current_end = int(group.loc[i, "end_frame_30fps_raw"])
        next_start = int(group.loc[i + 1, "start_frame_30fps_raw"])

        adjacency_rows.append({
            "sequence_id": sequence_id,
            "current_end": current_end,
            "next_start": next_start,
            "next_start_minus_current_end": next_start - current_end,
        })

adjacency = pd.DataFrame(adjacency_rows)

delta_counts = (
    adjacency["next_start_minus_current_end"]
    .value_counts()
    .sort_index()
    if len(adjacency)
    else pd.Series(dtype=int)
)

print("Most common adjacent-segment deltas:")
display(
    delta_counts.rename_axis("delta").reset_index(name="count").head(30)
)

if END_FRAME_CONVENTION == "auto":
    count_delta_0 = int((adjacency["next_start_minus_current_end"] == 0).sum())
    count_delta_1 = int((adjacency["next_start_minus_current_end"] == 1).sum())

    inferred_end_convention = (
        "inclusive"
        if count_delta_1 > count_delta_0
        else "exclusive"
    )
else:
    inferred_end_convention = END_FRAME_CONVENTION

if inferred_end_convention not in {"inclusive", "exclusive"}:
    raise ValueError(
        f"Unsupported END_FRAME_CONVENTION={END_FRAME_CONVENTION!r}"
    )

print("Configured convention:", END_FRAME_CONVENTION)
print("Selected convention:", inferred_end_convention)

segments = segments_raw.copy()

if inferred_end_convention == "inclusive":
    segments["end_frame_30fps_exclusive"] = (
        segments["end_frame_30fps_raw"] + 1
    )
else:
    segments["end_frame_30fps_exclusive"] = (
        segments["end_frame_30fps_raw"]
    )

segments["start_frame_30fps"] = segments["start_frame_30fps_raw"].astype(int)
segments["end_frame_30fps_exclusive"] = (
    segments["end_frame_30fps_exclusive"].astype(int)
)

bad_duration = (
    segments["end_frame_30fps_exclusive"]
    <= segments["start_frame_30fps"]
)

if bad_duration.any():
    display(segments.loc[bad_duration].head(50))
    raise ValueError("At least one segment has non-positive duration.")

Most common adjacent-segment deltas:


,delta,count
0,0,8073


Configured convention: auto
Selected convention: exclusive


## 10. Inspect and parse the official train/validation/test split files

In [10]:
official_split_paths = {
    ("train", "assembly"): COARSE_SPLIT_DIR / "train_coarse_assembly.txt",
    ("train", "disassembly"): COARSE_SPLIT_DIR / "train_coarse_disassembly.txt",
    ("val", "assembly"): COARSE_SPLIT_DIR / "val_coarse_assembly.txt",
    ("val", "disassembly"): COARSE_SPLIT_DIR / "val_coarse_disassembly.txt",
    ("test", "assembly"): COARSE_SPLIT_DIR / "test_coarse_assembly.txt",
    ("test", "disassembly"): COARSE_SPLIT_DIR / "test_coarse_disassembly.txt",
}

for key, path in official_split_paths.items():
    print("\n" + "=" * 100)
    print(key, path)
    print("=" * 100)

    if not path.exists():
        raise FileNotFoundError(path)

    for line in nonempty_lines(path)[:10]:
        print(repr(line))


('train', 'assembly') /content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations/coarse-annotations/coarse_splits/train_coarse_assembly.txt
'assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt\t\t-\tb06b\t-'
'assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.txt\t\t-\tb08c\t-'
'assembly_nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904.txt\t\t-\ta16\t-'
'assembly_nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713.txt\t\t-\tb06d\t-'
'assembly_nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034.txt\t\t-\tc06d\t-'
'assembly_nusar-2021_action_both_9012-c07c_9012_user_id_2021-02-01_164345.txt\t\t-\tc07c\t-'
'assembly_nusar-2021_action_both_9013-a02_9013_user_id_2021-02-02_130807.txt\t\t-\ta02\t-'
'assembly_nusar-2021_action_both_9013-b01a_9013_user_id_2021-02-02_135446.txt\t\t-\tb01a\t-'
'assembly_nusar-2021_action_both_9013-c03b_9013_user_id_2021-02-24_113410.txt\t\t-\tc03b\t-'
'assembly_nusar-2021_

## 11. Build the canonical official split manifest

In [11]:
def parse_split_file(path: Path, split_name: str, activity: str):
    rows = []

    for line_number, line in enumerate(nonempty_lines(path), start=1):
        # Train/val usually have:
        # sequence is_shared toy_id toy_name
        # Test usually has:
        # sequence is_shared
        parts = line.split(maxsplit=3)

        if len(parts) < 2:
            raise ValueError(
                f"Malformed split line {path.name}:{line_number}: {line!r}"
            )

        sequence_filename = parts[0]
        is_shared = parts[1]
        toy_id = parts[2] if len(parts) >= 3 else None
        toy_name = parts[3] if len(parts) >= 4 else None

        sequence_id = Path(sequence_filename).stem
        parsed_activity, recording_name = parse_activity_and_recording(sequence_id)

        if parsed_activity != activity:
            raise ValueError(
                f"Activity mismatch in {path.name}:{line_number}: "
                f"expected {activity}, got {parsed_activity}"
            )

        rows.append({
            "sequence_id": sequence_id,
            "sequence_filename": f"{sequence_id}.txt",
            "split": split_name,
            "activity": activity,
            "recording_name": recording_name,
            "is_shared": is_shared,
            "toy_id": toy_id,
            "toy_name": toy_name,
            "source_split_file": path.name,
            "source_split_line": line_number,
        })

    return rows


split_rows = []

for (split_name, activity), path in official_split_paths.items():
    split_rows.extend(
        parse_split_file(
            path=path,
            split_name=split_name,
            activity=activity,
        )
    )

split_manifest = pd.DataFrame(split_rows)

if split_manifest["sequence_id"].duplicated().any():
    duplicated = split_manifest[
        split_manifest["sequence_id"].duplicated(keep=False)
    ].sort_values("sequence_id")
    display(duplicated)
    raise ValueError(
        "A sequence appears more than once in the official split files."
    )

print("Official split counts:")
display(
    split_manifest
    .groupby(["split", "activity"])
    .size()
    .rename("num_sequences")
    .reset_index()
)

print("\nOverall split counts:")
display(
    split_manifest["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="num_sequences")
)

print("\nShared/not-shared counts:")
display(
    split_manifest
    .groupby(["split", "is_shared"])
    .size()
    .rename("num_sequences")
    .reset_index()
)

display(split_manifest.head(20))

Official split counts:


,split,activity,num_sequences
0,test,assembly,83
1,test,disassembly,84
2,train,assembly,191
3,train,disassembly,202
4,val,assembly,60
5,val,disassembly,60



Overall split counts:


,split,num_sequences
0,train,393
1,test,167
2,val,120



Shared/not-shared counts:


,split,is_shared,num_sequences
0,test,notshared,125
1,test,shared,42
2,train,-,393
3,val,notshared,82
4,val,shared,38


,sequence_id,sequence_filename,split,activity,recording_name,is_shared,toy_id,toy_name,source_split_file,source_split_line
0,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt,train,assembly,nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,-,b06b,-,train_coarse_assembly.txt,1
1,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.txt,train,assembly,nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,-,b08c,-,train_coarse_assembly.txt,2
2,assembly_nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904,assembly_nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904.txt,train,assembly,nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904,-,a16,-,train_coarse_assembly.txt,3
3,assembly_nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713,assembly_nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713.txt,train,assembly,nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713,-,b06d,-,train_coarse_assembly.txt,4
4,assembly_nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034,assembly_nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034.txt,train,assembly,nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034,-,c06d,-,train_coarse_assembly.txt,5
5,assembly_nusar-2021_action_both_9012-c07c_9012_user_id_2021-02-01_164345,assembly_nusar-2021_action_both_9012-c07c_9012_user_id_2021-02-01_164345.txt,train,assembly,nusar-2021_action_both_9012-c07c_9012_user_id_2021-02-01_164345,-,c07c,-,train_coarse_assembly.txt,6
6,assembly_nusar-2021_action_both_9013-a02_9013_user_id_2021-02-02_130807,assembly_nusar-2021_action_both_9013-a02_9013_user_id_2021-02-02_130807.txt,train,assembly,nusar-2021_action_both_9013-a02_9013_user_id_2021-02-02_130807,-,a02,-,train_coarse_assembly.txt,7
7,assembly_nusar-2021_action_both_9013-b01a_9013_user_id_2021-02-02_135446,assembly_nusar-2021_action_both_9013-b01a_9013_user_id_2021-02-02_135446.txt,train,assembly,nusar-2021_action_both_9013-b01a_9013_user_id_2021-02-02_135446,-,b01a,-,train_coarse_assembly.txt,8
8,assembly_nusar-2021_action_both_9013-c03b_9013_user_id_2021-02-24_113410,assembly_nusar-2021_action_both_9013-c03b_9013_user_id_2021-02-24_113410.txt,train,assembly,nusar-2021_action_both_9013-c03b_9013_user_id_2021-02-24_113410,-,c03b,-,train_coarse_assembly.txt,9
9,assembly_nusar-2021_action_both_9014-a12_9014_user_id_2021-02-02_141945,assembly_nusar-2021_action_both_9014-a12_9014_user_id_2021-02-02_141945.txt,train,assembly,nusar-2021_action_both_9014-a12_9014_user_id_2021-02-02_141945,-,a12,-,train_coarse_assembly.txt,10


## 12. Validate split/annotation coverage

In [12]:
annotation_sequence_set = set(segments["sequence_id"])
split_sequence_set = set(split_manifest["sequence_id"])

missing_annotations = sorted(split_sequence_set - annotation_sequence_set)
annotations_outside_splits = sorted(annotation_sequence_set - split_sequence_set)

print("Split sequences without annotation files:", len(missing_annotations))
print("Annotation files absent from official splits:", len(annotations_outside_splits))

if missing_annotations:
    print("\nFirst missing annotations:")
    for value in missing_annotations[:100]:
        print(value)
    raise ValueError(
        "At least one official split sequence has no downloaded coarse label file."
    )

if MAX_SEQUENCES is None and annotations_outside_splits:
    print("\nWARNING: annotation files outside official train/val/test splits:")
    for value in annotations_outside_splits[:50]:
        print(value)

split_sets = {
    split_name: set(
        split_manifest.loc[
            split_manifest["split"] == split_name,
            "sequence_id",
        ]
    )
    for split_name in ["train", "val", "test"]
}

overlap_train_val = split_sets["train"] & split_sets["val"]
overlap_train_test = split_sets["train"] & split_sets["test"]
overlap_val_test = split_sets["val"] & split_sets["test"]

print("train/val overlap:", len(overlap_train_val))
print("train/test overlap:", len(overlap_train_test))
print("val/test overlap:", len(overlap_val_test))

assert not overlap_train_val
assert not overlap_train_test
assert not overlap_val_test

Split sequences without annotation files: 0
Annotation files absent from official splits: 0
train/val overlap: 0
train/test overlap: 0
val/test overlap: 0


## 13. Parse `coarse_seq_views.txt` and validate selected-view availability

In [13]:
view_rows = []

for line_number, line in enumerate(
    nonempty_lines(COARSE_SEQ_VIEWS_PATH),
    start=1,
):
    normalized = line.replace("\\", "/").strip()
    parts = normalized.split("/")

    if len(parts) < 2:
        raise ValueError(
            f"Malformed view-list line {line_number}: {line!r}"
        )

    sequence_id = parts[-2]
    view_filename = parts[-1]

    activity, recording_name = parse_activity_and_recording(sequence_id)

    view_rows.append({
        "sequence_id": sequence_id,
        "activity": activity,
        "recording_name": recording_name,
        "view_filename": view_filename,
        "listed_path": normalized,
        "source_line": line_number,
    })

sequence_views = pd.DataFrame(view_rows)

print("Rows in coarse_seq_views.txt:", len(sequence_views))
print("Sequences represented:", sequence_views["sequence_id"].nunique())

display(
    sequence_views["view_filename"]
    .value_counts()
    .head(30)
    .rename_axis("view_filename")
    .reset_index(name="num_sequences")
)

selected_view_rows = sequence_views[
    sequence_views["view_filename"] == SELECTED_CAMERA_FILE
].copy()

selected_view_sequence_set = set(selected_view_rows["sequence_id"])
missing_selected_view = sorted(
    split_sequence_set - selected_view_sequence_set
)

print(
    f"Sequences with {SELECTED_CAMERA_FILE}:",
    len(selected_view_sequence_set),
)
print(
    "Official split sequences missing selected view:",
    len(missing_selected_view),
)

if missing_selected_view:
    print("First missing selected-view sequences:")
    for value in missing_selected_view[:100]:
        print(value)
    raise ValueError(
        f"Selected view {SELECTED_CAMERA_FILE} is unavailable "
        "for some official split sequences."
    )

Rows in coarse_seq_views.txt: 8070
Sequences represented: 680


,view_filename,num_sequences
0,C10095_rgb.mp4,680
1,C10115_rgb.mp4,680
2,C10118_rgb.mp4,680
3,C10119_rgb.mp4,680
4,C10379_rgb.mp4,680
5,C10390_rgb.mp4,680
6,C10395_rgb.mp4,680
7,C10404_rgb.mp4,680
8,HMC_84347414_mono10bit.mp4,332
9,HMC_84358933_mono10bit.mp4,330


Sequences with C10095_rgb.mp4: 680
Official split sequences missing selected view: 0


## 14. Build canonical sequence crops and detect overlaps/gaps

In [14]:
sequence_rows = []
overlap_rows = []
gap_rows = []

for sequence_id, group in segments.groupby("sequence_id", sort=True):
    group = group.sort_values(
        ["start_frame_30fps", "end_frame_30fps_exclusive"]
    ).reset_index(drop=True)

    activity = group["activity"].iloc[0]
    recording_name = group["recording_name"].iloc[0]

    clip_start = int(group["start_frame_30fps"].min())
    clip_end = int(group["end_frame_30fps_exclusive"].max())

    if clip_end <= clip_start:
        raise ValueError(f"Invalid clip range for {sequence_id}")

    conflicting_overlap_frames = 0
    same_label_overlap_frames = 0
    gap_frames = 0

    previous_end = None
    previous_label = None

    for row in group.itertuples(index=False):
        start = int(row.start_frame_30fps)
        end = int(row.end_frame_30fps_exclusive)
        label = row.action_cls

        if previous_end is not None:
            if start < previous_end:
                overlap = previous_end - start

                if label == previous_label:
                    same_label_overlap_frames += overlap
                else:
                    conflicting_overlap_frames += overlap

                overlap_rows.append({
                    "sequence_id": sequence_id,
                    "previous_end": previous_end,
                    "current_start": start,
                    "overlap_frames": overlap,
                    "previous_label": previous_label,
                    "current_label": label,
                    "conflicting": label != previous_label,
                })

            elif start > previous_end:
                gap = start - previous_end
                gap_frames += gap

                gap_rows.append({
                    "sequence_id": sequence_id,
                    "gap_start_frame_30fps": previous_end,
                    "gap_end_frame_30fps_exclusive": start,
                    "gap_frames": gap,
                })

        previous_end = max(previous_end or end, end)
        previous_label = label

    sequence_rows.append({
        "sequence_id": sequence_id,
        "sequence_filename": f"{sequence_id}.txt",
        "activity": activity,
        "recording_name": recording_name,
        "clip_start_frame_30fps": clip_start,
        "clip_end_frame_30fps_exclusive": clip_end,
        "clip_num_frames_30fps": clip_end - clip_start,
        "clip_start_seconds": clip_start / ANNOTATION_FPS,
        "clip_end_seconds": clip_end / ANNOTATION_FPS,
        "clip_duration_seconds": (clip_end - clip_start) / ANNOTATION_FPS,
        "num_action_segments": len(group),
        "num_unique_action_classes": group["action_cls"].nunique(),
        "gap_frames_inside_clip": gap_frames,
        "same_label_overlap_frames": same_label_overlap_frames,
        "conflicting_overlap_frames": conflicting_overlap_frames,
    })

sequence_manifest = pd.DataFrame(sequence_rows)

overlap_manifest = pd.DataFrame(overlap_rows)
gap_manifest = pd.DataFrame(gap_rows)

print("Sequences:", len(sequence_manifest))
print("Total segments:", len(segments))
print("Total internal gap frames:", int(sequence_manifest["gap_frames_inside_clip"].sum()))
print("Total same-label overlap frames:", int(sequence_manifest["same_label_overlap_frames"].sum()))
print("Total conflicting overlap frames:", int(sequence_manifest["conflicting_overlap_frames"].sum()))

display(sequence_manifest.head(20))

if FAIL_ON_CONFLICTING_OVERLAPS:
    conflicting_total = int(
        sequence_manifest["conflicting_overlap_frames"].sum()
    )
    if conflicting_total > 0:
        display(
            overlap_manifest[
                overlap_manifest["conflicting"]
            ].head(100)
        )
        raise ValueError(
            "Conflicting overlapping coarse segments were found."
        )

Sequences: 680
Total segments: 8753
Total internal gap frames: 0
Total same-label overlap frames: 0
Total conflicting overlap frames: 0


,sequence_id,sequence_filename,activity,recording_name,clip_start_frame_30fps,clip_end_frame_30fps_exclusive,clip_num_frames_30fps,clip_start_seconds,clip_end_seconds,clip_duration_seconds,num_action_segments,num_unique_action_classes,gap_frames_inside_clip,same_label_overlap_frames,conflicting_overlap_frames
0,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,assembly,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,4457,8070,3613,148.566667,269.000000,120.433333,9,8,0,0,0
1,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt,assembly,nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,2833,6959,4126,94.433333,231.966667,137.533333,7,7,0,0,0
2,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.txt,assembly,nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,4777,11525,6748,159.233333,384.166667,224.933333,14,14,0,0,0
3,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620.txt,assembly,nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,4380,9978,5598,146.000000,332.600000,186.600000,14,14,0,0,0
4,assembly_nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,assembly_nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239.txt,assembly,nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,2916,6755,3839,97.200000,225.166667,127.966667,10,10,0,0,0
5,assembly_nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915,assembly_nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915.txt,assembly,nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915,3680,10403,6723,122.666667,346.766667,224.100000,20,15,0,0,0
6,assembly_nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904,assembly_nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904.txt,assembly,nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904,5222,12863,7641,174.066667,428.766667,254.700000,18,18,0,0,0
7,assembly_nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209,assembly_nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209.txt,assembly,nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209,4013,10597,6584,133.766667,353.233333,219.466667,18,15,0,0,0
8,assembly_nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713,assembly_nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713.txt,assembly,nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713,3853,8134,4281,128.433333,271.133333,142.700000,7,7,0,0,0
9,assembly_nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034,assembly_nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034.txt,assembly,nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034,3650,8698,5048,121.666667,289.933333,168.266667,15,13,0,0,0


## 15. Join official split metadata to sequences

In [15]:
sequence_manifest = sequence_manifest.merge(
    split_manifest,
    on=[
        "sequence_id",
        "sequence_filename",
        "activity",
        "recording_name",
    ],
    how="left",
    validate="one_to_one",
)

missing_split_assignment = sequence_manifest["split"].isna()

print(
    "Sequences without official split assignment:",
    int(missing_split_assignment.sum()),
)

if MAX_SEQUENCES is None and missing_split_assignment.any():
    display(
        sequence_manifest.loc[
            missing_split_assignment,
            ["sequence_id", "activity", "recording_name"],
        ].head(100)
    )
    raise ValueError(
        "Some converted sequences have no official split assignment."
    )

# In debug mode, retain only sequences available in the truncated conversion.
if MAX_SEQUENCES is not None:
    sequence_manifest = sequence_manifest[
        ~missing_split_assignment
    ].copy()

segments = segments[
    segments["sequence_id"].isin(
        set(sequence_manifest["sequence_id"])
    )
].copy()

display(
    sequence_manifest
    .groupby(["split", "activity"])
    .agg(
        num_sequences=("sequence_id", "size"),
        total_duration_hours=(
            "clip_duration_seconds",
            lambda x: float(x.sum()) / 3600.0,
        ),
        total_segments=("num_action_segments", "sum"),
    )
    .reset_index()
)

Sequences without official split assignment: 0


,split,activity,num_sequences,total_duration_hours,total_segments
0,test,assembly,83,6.078417,1415
1,test,disassembly,84,4.088167,929
2,train,assembly,191,13.393389,2761
3,train,disassembly,202,8.962843,2059
4,val,assembly,60,4.809352,953
5,val,disassembly,60,2.875361,636


## 16. Join the remote `v1` recording paths from notebook 12

In [16]:
recording_inventory = pd.read_csv(RECORDING_INVENTORY_PATH)

required_recording_columns = {
    "recording_name",
    "filename",
    "remote_path",
    "view",
}

missing_recording_columns = (
    required_recording_columns
    - set(recording_inventory.columns)
)

if missing_recording_columns:
    raise ValueError(
        "Recording inventory is missing columns: "
        f"{sorted(missing_recording_columns)}"
    )

selected_recordings = recording_inventory[
    (recording_inventory["view"] == SELECTED_VIEW)
    & (recording_inventory["filename"] == SELECTED_CAMERA_FILE)
].copy()

if selected_recordings["recording_name"].duplicated().any():
    display(
        selected_recordings[
            selected_recordings["recording_name"].duplicated(keep=False)
        ]
    )
    raise ValueError(
        "Duplicate selected-view entries for a recording."
    )

selected_recordings = selected_recordings[
    ["recording_name", "filename", "remote_path", "view"]
].rename(
    columns={
        "filename": "video_filename",
        "remote_path": "video_remote_path",
        "view": "selected_view",
    }
)

sequence_video_manifest = sequence_manifest.merge(
    selected_recordings,
    on="recording_name",
    how="left",
    validate="many_to_one",
)

sequence_video_manifest["video_local_path"] = (
    sequence_video_manifest["video_remote_path"]
    .fillna("")
    .map(
        lambda value: (
            str(ASSEMBLY_ROOT / value)
            if value
            else ""
        )
    )
)

sequence_video_manifest["video_downloaded"] = (
    sequence_video_manifest["video_local_path"]
    .map(lambda value: bool(value) and Path(value).exists())
)

missing_remote_video = (
    sequence_video_manifest["video_remote_path"].isna()
)

print(
    "Sequence crops with remote selected-view path:",
    int((~missing_remote_video).sum()),
    "/",
    len(sequence_video_manifest),
)
print(
    "Sequence crops with selected-view video already downloaded:",
    int(sequence_video_manifest["video_downloaded"].sum()),
)

if missing_remote_video.any():
    display(
        sequence_video_manifest.loc[
            missing_remote_video,
            ["sequence_id", "recording_name"],
        ].head(100)
    )
    raise ValueError(
        "Some sequence crops cannot be matched to the remote v1 video."
    )

display(
    sequence_video_manifest[
        [
            "sequence_id",
            "split",
            "activity",
            "recording_name",
            "video_remote_path",
            "clip_start_seconds",
            "clip_end_seconds",
            "video_downloaded",
        ]
    ].head(20)
)

Sequence crops with remote selected-view path: 680 / 680
Sequence crops with selected-view video already downloaded: 0


,sequence_id,split,activity,recording_name,video_remote_path,clip_start_seconds,clip_end_seconds,video_downloaded
0,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,test,assembly,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,recordings/nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724/C10095_rgb.mp4,148.566667,269.000000,False
1,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,train,assembly,nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,recordings/nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253/C10095_rgb.mp4,94.433333,231.966667,False
2,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,train,assembly,nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,recordings/nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736/C10095_rgb.mp4,159.233333,384.166667,False
3,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,test,assembly,nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,recordings/nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620/C10095_rgb.mp4,146.000000,332.600000,False
4,assembly_nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,val,assembly,nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,recordings/nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239/C10095_rgb.mp4,97.200000,225.166667,False
5,assembly_nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915,test,assembly,nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915,recordings/nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915/C10095_rgb.mp4,122.666667,346.766667,False
6,assembly_nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904,train,assembly,nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904,recordings/nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904/C10095_rgb.mp4,174.066667,428.766667,False
7,assembly_nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209,test,assembly,nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209,recordings/nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209/C10095_rgb.mp4,133.766667,353.233333,False
8,assembly_nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713,train,assembly,nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713,recordings/nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713/C10095_rgb.mp4,128.433333,271.133333,False
9,assembly_nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034,train,assembly,nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034,recordings/nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034/C10095_rgb.mp4,121.666667,289.933333,False


## 17. Create dense 30 fps ground-truth files

In [17]:
# Remove only previously generated text labels in this dedicated output folder.
for old_path in GT_DIR.glob("*.txt"):
    old_path.unlink()

ground_truth_rows = []
conflicting_write_rows = []

sequence_lookup = sequence_manifest.set_index("sequence_id")

for sequence_id, group in segments.groupby("sequence_id", sort=True):
    meta = sequence_lookup.loc[sequence_id]

    clip_start = int(meta["clip_start_frame_30fps"])
    clip_end = int(meta["clip_end_frame_30fps_exclusive"])
    clip_length = clip_end - clip_start

    dense = np.empty(clip_length, dtype=object)
    dense[:] = None

    for row in group.sort_values(
        ["start_frame_30fps", "end_frame_30fps_exclusive"]
    ).itertuples(index=False):
        local_start = int(row.start_frame_30fps) - clip_start
        local_end = int(row.end_frame_30fps_exclusive) - clip_start
        label = row.action_cls

        if not (0 <= local_start < local_end <= clip_length):
            raise ValueError(
                f"Segment outside clip bounds for {sequence_id}: "
                f"[{local_start}, {local_end}) vs length {clip_length}"
            )

        existing = dense[local_start:local_end]
        occupied = existing != None  # noqa: E711

        if occupied.any():
            conflicting_mask = occupied & (existing != label)

            if conflicting_mask.any():
                conflicting_write_rows.append({
                    "sequence_id": sequence_id,
                    "action_cls": label,
                    "local_start": local_start,
                    "local_end": local_end,
                    "num_conflicting_frames": int(conflicting_mask.sum()),
                })

                if FAIL_ON_CONFLICTING_OVERLAPS:
                    raise ValueError(
                        f"Conflicting labels while writing {sequence_id}"
                    )

        # Later segment fills empty positions and overwrites only same-label overlaps.
        empty_or_same = (existing == None) | (existing == label)  # noqa: E711
        existing[empty_or_same] = label
        dense[local_start:local_end] = existing

    num_background_frames = int(
        sum(value is None for value in dense)
    )

    if num_background_frames:
        dense = np.array(
            [
                BACKGROUND_LABEL if value is None else value
                for value in dense
            ],
            dtype=object,
        )

    gt_path = GT_DIR / f"{sequence_id}.txt"
    gt_path.write_text(
        "\n".join(map(str, dense.tolist())) + "\n",
        encoding="utf-8",
    )

    ground_truth_rows.append({
        "sequence_id": sequence_id,
        "ground_truth_path": str(gt_path),
        "ground_truth_length_30fps": len(dense),
        "num_background_frames": num_background_frames,
        "background_fraction": (
            num_background_frames / len(dense)
            if len(dense)
            else 0.0
        ),
        "first_label": dense[0] if len(dense) else None,
        "last_label": dense[-1] if len(dense) else None,
    })

ground_truth_manifest = pd.DataFrame(ground_truth_rows)

print("Written ground-truth files:", len(ground_truth_manifest))
print("Total dense labels:", int(ground_truth_manifest["ground_truth_length_30fps"].sum()))
print("Total background frames:", int(ground_truth_manifest["num_background_frames"].sum()))
print(
    "Sequences containing background:",
    int((ground_truth_manifest["num_background_frames"] > 0).sum()),
)

display(ground_truth_manifest.head(20))

Written ground-truth files: 680
Total dense labels: 4342413
Total background frames: 0
Sequences containing background: 0


,sequence_id,ground_truth_path,ground_truth_length_30fps,num_background_frames,background_fraction,first_label,last_label
0,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-a01_9011_user_id_...,3613,0,0.0,attach chassis,attach track
1,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-b06b_9011_user_id...,4126,0,0.0,attach interior,demonstrate functionality
2,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-b08c_9011_user_id...,6748,0,0.0,attach base,demonstrate functionality
3,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-c01c_9011_user_id...,5598,0,0.0,attach wheel,demonstrate functionality
4,assembly_nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-c03f_9011_user_id...,3839,0,0.0,attach chassis,demonstrate functionality
5,assembly_nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-c13b_9011_user_id...,6723,0,0.0,attach wheel,demonstrate functionality
6,assembly_nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9012-a16_9012_user_id_...,7641,0,0.0,attach wheel,demonstrate functionality
7,assembly_nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9012-a17_9012_user_id_...,6584,0,0.0,attach wheel,demonstrate functionality
8,assembly_nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9012-b06d_9012_user_id...,4281,0,0.0,attach interior,demonstrate functionality
9,assembly_nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9012-c06d_9012_user_id...,5048,0,0.0,attach wheel,demonstrate functionality


## 18. Build `mapping.txt` and class metadata

In [18]:
background_is_used = bool(
    (ground_truth_manifest["num_background_frames"] > 0).any()
)

class_metadata = actions.copy()

# Preserve official action IDs in metadata, while producing contiguous
# model IDs in mapping.txt.
class_metadata = class_metadata.rename(
    columns={"action_id": "official_action_id"}
)

class_metadata["model_class_id"] = np.arange(
    len(class_metadata),
    dtype=int,
)
class_metadata["is_background"] = False

if background_is_used:
    background_row = pd.DataFrame([{
        "official_action_id": np.nan,
        "verb_id": np.nan,
        "noun_id": np.nan,
        "action_cls": BACKGROUND_LABEL,
        "verb_cls": BACKGROUND_LABEL,
        "noun_cls": BACKGROUND_LABEL,
        "model_class_id": len(class_metadata),
        "is_background": True,
    }])

    class_metadata = pd.concat(
        [class_metadata, background_row],
        ignore_index=True,
    )

class_metadata["model_class_id"] = (
    class_metadata["model_class_id"].astype(int)
)

mapping_lines = [
    f"{int(row.model_class_id)} {row.action_cls}"
    for row in class_metadata.itertuples(index=False)
]

mapping_path = OUT_ROOT / "mapping.txt"
mapping_path.write_text(
    "\n".join(mapping_lines) + "\n",
    encoding="utf-8",
)

class_metadata_path = OUT_ROOT / "class_metadata.csv"
class_metadata.to_csv(class_metadata_path, index=False)

print("Background class included:", background_is_used)
print("Number of model classes:", len(class_metadata))
print("Saved:", mapping_path)
print("Saved:", class_metadata_path)

print("\nFirst mapping entries:")
print("\n".join(mapping_lines[:20]))

print("\nLast mapping entries:")
print("\n".join(mapping_lines[-10:]))

Background class included: False
Number of model classes: 202
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/mapping.txt
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/class_metadata.csv

First mapping entries:
0 inspect toy
1 attach cabin
2 detach cabin
3 detach wheel
4 attach wheel
5 screw chassis
6 demonstrate functionality
7 unscrew chassis
8 attach interior
9 detach roof
10 attach roof
11 detach interior
12 attach bumper
13 attach body
14 detach bumper
15 detach body
16 attach base
17 detach base
18 unscrew interior
19 attach arm

Last mapping entries:
192 attempt to attach rocker panel
193 screw roller
194 unscrew engine cover
195 screw mixer stand
196 screw grill
197 attempt to attach strap
198 attach fire equipment
199 detach battery
200 attach battery
201 attempt to attach sound module


## 19. Write MS-TCN split bundles

In [19]:
available_sequence_set = set(sequence_manifest["sequence_id"])

bundle_paths = {}

for split_name in ["train", "val", "test"]:
    sequence_ids = sorted(
        sequence_manifest.loc[
            sequence_manifest["split"] == split_name,
            "sequence_id",
        ]
    )

    bundle_path = (
        SPLIT_OUT_DIR
        / f"{split_name}.split{SPLIT_ID}.bundle"
    )

    bundle_path.write_text(
        "\n".join(f"{sequence_id}.txt" for sequence_id in sequence_ids)
        + "\n",
        encoding="utf-8",
    )

    bundle_paths[split_name] = bundle_path
    print(split_name, len(sequence_ids), "->", bundle_path)

train_val_ids = sorted(
    sequence_manifest.loc[
        sequence_manifest["split"].isin(["train", "val"]),
        "sequence_id",
    ]
)

train_val_bundle_path = (
    SPLIT_OUT_DIR
    / f"train_val.split{SPLIT_ID}.bundle"
)

train_val_bundle_path.write_text(
    "\n".join(f"{sequence_id}.txt" for sequence_id in train_val_ids)
    + "\n",
    encoding="utf-8",
)

bundle_paths["train_val"] = train_val_bundle_path

print(
    "train_val",
    len(train_val_ids),
    "->",
    train_val_bundle_path,
)

train 393 -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/splits/train.split1.bundle
val 120 -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/splits/val.split1.bundle
test 167 -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/splits/test.split1.bundle
train_val 513 -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/splits/train_val.split1.bundle


## 20. Create download and ProcedureVRL extraction manifests

This section explicitly merges the ground-truth metadata produced in section 17 into the video manifest before selecting the `ground_truth_*` columns.

In [20]:
# Attach the ground-truth metadata created in section 17.
# sequence_video_manifest was originally built before ground_truth_manifest,
# so these columns are not present until we merge them explicitly.
gt_columns = [
    "sequence_id",
    "ground_truth_path",
    "ground_truth_length_30fps",
    "num_background_frames",
    "background_fraction",
]

missing_gt_columns = [
    column for column in gt_columns
    if column not in ground_truth_manifest.columns
]

if missing_gt_columns:
    raise KeyError(
        "ground_truth_manifest is missing required columns: "
        f"{missing_gt_columns}"
    )

# Make the cell safe to rerun: remove previously merged GT columns first.
sequence_video_manifest = sequence_video_manifest.drop(
    columns=[
        column for column in gt_columns[1:]
        if column in sequence_video_manifest.columns
    ],
    errors="ignore",
)

sequence_video_manifest = sequence_video_manifest.merge(
    ground_truth_manifest[gt_columns],
    on="sequence_id",
    how="left",
    validate="one_to_one",
)

missing_gt_rows = sequence_video_manifest[
    "ground_truth_path"
].isna()

print(
    "Sequence rows with attached ground-truth metadata:",
    int((~missing_gt_rows).sum()),
    "/",
    len(sequence_video_manifest),
)

if missing_gt_rows.any():
    display(
        sequence_video_manifest.loc[
            missing_gt_rows,
            ["sequence_id", "recording_name", "split"],
        ].head(100)
    )
    raise ValueError(
        "Some sequence/video rows could not be matched to "
        "ground_truth_manifest."
    )

recording_download_manifest = (
    sequence_video_manifest[
        [
            "recording_name",
            "selected_view",
            "video_filename",
            "video_remote_path",
            "video_local_path",
            "video_downloaded",
        ]
    ]
    .drop_duplicates(subset=["recording_name"])
    .sort_values("recording_name")
    .reset_index(drop=True)
)

sequence_crop_counts = (
    sequence_video_manifest
    .groupby("recording_name")
    .size()
    .rename("num_sequence_crops")
    .reset_index()
)

recording_download_manifest = recording_download_manifest.merge(
    sequence_crop_counts,
    on="recording_name",
    how="left",
    validate="one_to_one",
)

procedurevrl_manifest = sequence_video_manifest[
    [
        "sequence_id",
        "sequence_filename",
        "split",
        "activity",
        "is_shared",
        "toy_id",
        "toy_name",
        "recording_name",
        "selected_view",
        "video_filename",
        "video_remote_path",
        "video_local_path",
        "video_downloaded",
        "clip_start_frame_30fps",
        "clip_end_frame_30fps_exclusive",
        "clip_start_seconds",
        "clip_end_seconds",
        "clip_duration_seconds",
        "ground_truth_path",
        "ground_truth_length_30fps",
        "num_background_frames",
    ]
].copy()

procedurevrl_manifest["output_feature_name"] = (
    procedurevrl_manifest["sequence_id"] + ".npy"
)

procedurevrl_manifest["feature_extraction_status"] = np.where(
    procedurevrl_manifest["video_downloaded"],
    "ready",
    "video_not_downloaded",
)

recording_download_path = (
    OUT_ROOT / f"recording_download_manifest_{SELECTED_VIEW}.csv"
)
procedurevrl_manifest_path = (
    OUT_ROOT / f"procedurevrl_extraction_manifest_{SELECTED_VIEW}.csv"
)
sequence_video_manifest_path = (
    OUT_ROOT / f"sequence_video_manifest_{SELECTED_VIEW}.csv"
)

recording_download_manifest.to_csv(
    recording_download_path,
    index=False,
)
procedurevrl_manifest.to_csv(
    procedurevrl_manifest_path,
    index=False,
)
sequence_video_manifest.to_csv(
    sequence_video_manifest_path,
    index=False,
)

print("Unique raw recordings required:", len(recording_download_manifest))
print("Sequence crops to extract:", len(procedurevrl_manifest))
print("Downloaded recordings:", int(recording_download_manifest["video_downloaded"].sum()))

print("\nSaved:")
print(recording_download_path)
print(procedurevrl_manifest_path)
print(sequence_video_manifest_path)

display(recording_download_manifest.head(20))
display(procedurevrl_manifest.head(20))

Sequence rows with attached ground-truth metadata: 680 / 680
Unique raw recordings required: 350
Sequence crops to extract: 680
Downloaded recordings: 0

Saved:
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/recording_download_manifest_v1.csv
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/procedurevrl_extraction_manifest_v1.csv
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/sequence_video_manifest_v1.csv


,recording_name,selected_view,video_filename,video_remote_path,video_local_path,video_downloaded,num_sequence_crops
0,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724/C10095_rgb.mp4,False,2
1,nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253/C10095_rgb.mp4,False,2
2,nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736/C10095_rgb.mp4,False,2
3,nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620/C10095_rgb.mp4,False,2
4,nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239/C10095_rgb.mp4,False,2
5,nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915/C10095_rgb.mp4,False,2
6,nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904/C10095_rgb.mp4,False,2
7,nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209/C10095_rgb.mp4,False,2
8,nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713/C10095_rgb.mp4,False,2
9,nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034/C10095_rgb.mp4,False,2


,sequence_id,sequence_filename,split,activity,is_shared,toy_id,toy_name,recording_name,selected_view,video_filename,video_remote_path,video_local_path,video_downloaded,clip_start_frame_30fps,clip_end_frame_30fps_exclusive,clip_start_seconds,clip_end_seconds,clip_duration_seconds,ground_truth_path,ground_truth_length_30fps,num_background_frames,output_feature_name,feature_extraction_status
0,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,test,assembly,notshared,None,None,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724/C10095_rgb.mp4,False,4457,8070,148.566667,269.000000,120.433333,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-a01_9011_user_id_...,3613,0,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.npy,video_not_downloaded
1,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt,train,assembly,-,b06b,-,nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253/C10095_rgb.mp4,False,2833,6959,94.433333,231.966667,137.533333,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-b06b_9011_user_id...,4126,0,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.npy,video_not_downloaded
2,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.txt,train,assembly,-,b08c,-,nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736/C10095_rgb.mp4,False,4777,11525,159.233333,384.166667,224.933333,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-b08c_9011_user_id...,6748,0,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.npy,video_not_downloaded
3,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620.txt,test,assembly,notshared,None,None,nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620/C10095_rgb.mp4,False,4380,9978,146.000000,332.600000,186.600000,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-c01c_9011_user_id...,5598,0,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620.npy,video_not_downloaded
4,assembly_nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,assembly_nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239.txt,val,assembly,notshared,c03f,roller,nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239/C10095_rgb.mp4,/content/drive/MyDr

## 21. Save canonical manifests

In [21]:
segments_out = segments.merge(
    sequence_manifest[
        [
            "sequence_id",
            "split",
            "clip_start_frame_30fps",
            "clip_end_frame_30fps_exclusive",
        ]
    ],
    on="sequence_id",
    how="left",
    validate="many_to_one",
)

segments_out["local_start_frame_30fps"] = (
    segments_out["start_frame_30fps"]
    - segments_out["clip_start_frame_30fps"]
)
segments_out["local_end_frame_30fps_exclusive"] = (
    segments_out["end_frame_30fps_exclusive"]
    - segments_out["clip_start_frame_30fps"]
)
segments_out["segment_duration_frames_30fps"] = (
    segments_out["local_end_frame_30fps_exclusive"]
    - segments_out["local_start_frame_30fps"]
)
segments_out["segment_duration_seconds"] = (
    segments_out["segment_duration_frames_30fps"]
    / ANNOTATION_FPS
)

segment_manifest_path = OUT_ROOT / "segment_manifest.csv"
sequence_manifest_path = OUT_ROOT / "sequence_manifest.csv"
split_manifest_path = OUT_ROOT / "official_split_manifest.csv"
ground_truth_manifest_path = OUT_ROOT / "ground_truth_manifest.csv"
overlap_manifest_path = OUT_ROOT / "overlap_manifest.csv"
gap_manifest_path = OUT_ROOT / "gap_manifest.csv"

segments_out.to_csv(segment_manifest_path, index=False)
sequence_manifest.to_csv(sequence_manifest_path, index=False)
split_manifest.to_csv(split_manifest_path, index=False)
ground_truth_manifest.to_csv(ground_truth_manifest_path, index=False)
overlap_manifest.to_csv(overlap_manifest_path, index=False)
gap_manifest.to_csv(gap_manifest_path, index=False)

print("Saved:")
for path in [
    segment_manifest_path,
    sequence_manifest_path,
    split_manifest_path,
    ground_truth_manifest_path,
    overlap_manifest_path,
    gap_manifest_path,
]:
    print(path)

Saved:
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/segment_manifest.csv
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/sequence_manifest.csv
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/official_split_manifest.csv
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/ground_truth_manifest.csv
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/overlap_manifest.csv
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/gap_manifest.csv


## 22. Validate the generated MS-TCN format

In [22]:
def read_nonempty(path: Path):
    return [
        line.strip()
        for line in path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
        if line.strip()
    ]


mapping_entries = {}

for line in read_nonempty(mapping_path):
    class_id_raw, class_label = line.split(maxsplit=1)
    class_id = int(class_id_raw)

    if class_id in mapping_entries:
        raise ValueError(f"Duplicate mapping ID: {class_id}")

    mapping_entries[class_id] = class_label

mapping_label_set = set(mapping_entries.values())

validation_rows = []

for row in sequence_manifest.itertuples(index=False):
    gt_path = GT_DIR / f"{row.sequence_id}.txt"
    labels = read_nonempty(gt_path)

    unknown = sorted(set(labels) - mapping_label_set)

    validation_rows.append({
        "sequence_id": row.sequence_id,
        "split": row.split,
        "activity": row.activity,
        "gt_exists": gt_path.exists(),
        "gt_length": len(labels),
        "expected_gt_length": int(row.clip_num_frames_30fps),
        "length_matches": (
            len(labels) == int(row.clip_num_frames_30fps)
        ),
        "num_unknown_labels": len(unknown),
        "num_unique_labels": len(set(labels)),
        "has_background": BACKGROUND_LABEL in labels,
    })

validation_report = pd.DataFrame(validation_rows)

print("Validation summary:")
print("Sequences:", len(validation_report))
print("Missing GT files:", int((~validation_report["gt_exists"]).sum()))
print("Length mismatches:", int((~validation_report["length_matches"]).sum()))
print("Unknown labels:", int(validation_report["num_unknown_labels"].sum()))
print("Unique GT lengths:", validation_report["gt_length"].nunique())

display(
    validation_report.groupby(["split", "activity"])
    .agg(
        num_sequences=("sequence_id", "size"),
        total_frames_30fps=("gt_length", "sum"),
        min_frames=("gt_length", "min"),
        max_frames=("gt_length", "max"),
        mean_frames=("gt_length", "mean"),
    )
    .reset_index()
)

assert validation_report["gt_exists"].all()
assert validation_report["length_matches"].all()
assert validation_report["num_unknown_labels"].sum() == 0

for split_name, bundle_path in bundle_paths.items():
    bundle_entries = read_nonempty(bundle_path)

    missing_bundle_gt = [
        entry
        for entry in bundle_entries
        if not (GT_DIR / entry).exists()
    ]

    print(
        f"{split_name}: entries={len(bundle_entries)}, "
        f"missing GT={len(missing_bundle_gt)}"
    )

    assert not missing_bundle_gt

validation_report_path = OUT_ROOT / "validation_report.csv"
validation_report.to_csv(validation_report_path, index=False)
print("Saved:", validation_report_path)

Validation summary:
Sequences: 680
Missing GT files: 0
Length mismatches: 0
Unknown labels: 0
Unique GT lengths: 649


,split,activity,num_sequences,total_frames_30fps,min_frames,max_frames,mean_frames
0,test,assembly,83,656469,2780,25943,7909.265060
1,test,disassembly,84,441522,2361,18287,5256.214286
2,train,assembly,191,1446486,2206,46657,7573.225131
3,train,disassembly,202,967987,1858,11565,4792.014851
4,val,assembly,60,519410,2649,23831,8656.833333
5,val,disassembly,60,310539,2140,12668,5175.650000


train: entries=393, missing GT=0
val: entries=120, missing GT=0
test: entries=167, missing GT=0
train_val: entries=513, missing GT=0
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/validation_report.csv


## 23. Dataset statistics

In [23]:
print("Class-frequency statistics from annotated segments:")

class_stats = (
    segments_out
    .groupby(
        ["official_action_id", "action_cls"],
        as_index=False,
    )
    .agg(
        num_segments=("sequence_id", "size"),
        total_frames_30fps=(
            "segment_duration_frames_30fps",
            "sum",
        ),
        num_sequences=("sequence_id", "nunique"),
    )
)

class_stats["total_duration_minutes"] = (
    class_stats["total_frames_30fps"]
    / ANNOTATION_FPS
    / 60.0
)

class_stats = class_stats.sort_values(
    ["total_frames_30fps", "num_segments"],
    ascending=False,
).reset_index(drop=True)

class_stats_path = OUT_ROOT / "class_statistics.csv"
class_stats.to_csv(class_stats_path, index=False)

display(class_stats.head(30))
display(class_stats.tail(30))

print("Saved:", class_stats_path)

print("\nSequence-duration statistics:")
display(
    sequence_manifest[
        [
            "clip_duration_seconds",
            "num_action_segments",
            "num_unique_action_classes",
            "gap_frames_inside_clip",
        ]
    ].describe()
)

Class-frequency statistics from annotated segments:


,official_action_id,action_cls,num_segments,total_frames_30fps,num_sequences,total_duration_minutes
0,4,attach wheel,342,488429,313,271.349444
1,3,detach wheel,353,431771,328,239.872778
2,0,inspect toy,584,206210,319,114.561111
3,5,screw chassis,333,175576,216,97.542222
4,7,unscrew chassis,313,151198,248,83.998889
5,10,attach roof,261,138988,213,77.215556
6,1,attach cabin,398,137286,294,76.270000
7,12,attach bumper,233,116724,182,64.846667
8,9,detach roof,263,108071,236,60.039444
9,8,attach interior,277,106749,221,59.305000


,official_action_id,action_cls,num_segments,total_frames_30fps,num_sequences,total_duration_minutes
172,171,screw arm connector,6,2021,6,1.122778
173,112,remove figurine,12,1975,11,1.097222
174,181,attempt to attach cylinder,5,1970,2,1.094444
175,187,unscrew lid,4,1960,4,1.088889
176,170,detach engine,6,1816,6,1.008889
177,190,detach fire equipment,3,1813,2,1.007222
178,177,inspect water tank,5,1774,5,0.985556
179,178,attach engine,5,1757,5,0.976111
180,179,attempt to attach roller arm,5,1608,4,0.893333
181,180,unscrew hook,5,1573,5,0.873889


Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/class_statistics.csv

Sequence-duration statistics:


,clip_duration_seconds,num_action_segments,num_unique_action_classes,gap_frames_inside_clip
count,680.000000,680.000000,680.000000,680.0
mean,212.863382,12.872059,10.536765,0.0
std,130.437333,7.001774,3.359001,0.0
min,61.933333,3.000000,3.000000,0.0
25%,135.900000,9.000000,8.000000,0.0
50%,180.033333,11.000000,10.000000,0.0
75%,250.250000,14.000000,12.000000,0.0
max,1555.233333,73.000000,25.000000,0.0


## 24. Save final summary

In [24]:
split_counts = {
    split_name: int(
        (sequence_manifest["split"] == split_name).sum()
    )
    for split_name in ["train", "val", "test"]
}

activity_counts = {
    activity: int(
        (sequence_manifest["activity"] == activity).sum()
    )
    for activity in ["assembly", "disassembly"]
}

summary = {
    "status": "completed",
    "dataset": "Assembly101",
    "annotation_granularity": "coarse",
    "task": "temporal action segmentation",
    "annotation_fps": ANNOTATION_FPS,
    "raw_video_fps_description": RAW_VIDEO_FPS_DESCRIPTION,
    "end_frame_convention": inferred_end_convention,
    "selected_view": SELECTED_VIEW,
    "selected_camera_file": SELECTED_CAMERA_FILE,
    "output_root": str(OUT_ROOT),
    "num_official_action_classes": int(len(actions)),
    "background_class_included": background_is_used,
    "num_model_classes": int(len(class_metadata)),
    "num_sequences": int(len(sequence_manifest)),
    "num_segments": int(len(segments_out)),
    "num_unique_recordings_required": int(
        len(recording_download_manifest)
    ),
    "num_recordings_already_downloaded": int(
        recording_download_manifest["video_downloaded"].sum()
    ),
    "split_counts": split_counts,
    "activity_counts": activity_counts,
    "total_duration_hours": float(
        sequence_manifest["clip_duration_seconds"].sum()
        / 3600.0
    ),
    "total_dense_frames_30fps": int(
        ground_truth_manifest[
            "ground_truth_length_30fps"
        ].sum()
    ),
    "total_background_frames": int(
        ground_truth_manifest[
            "num_background_frames"
        ].sum()
    ),
    "total_conflicting_overlap_frames": int(
        sequence_manifest[
            "conflicting_overlap_frames"
        ].sum()
    ),
    "validation": {
        "missing_ground_truth_files": int(
            (~validation_report["gt_exists"]).sum()
        ),
        "ground_truth_length_mismatches": int(
            (~validation_report["length_matches"]).sum()
        ),
        "unknown_ground_truth_labels": int(
            validation_report[
                "num_unknown_labels"
            ].sum()
        ),
        "train_val_overlap": len(overlap_train_val),
        "train_test_overlap": len(overlap_train_test),
        "val_test_overlap": len(overlap_val_test),
        "split_sequences_missing_annotations": len(
            missing_annotations
        ),
        "split_sequences_missing_selected_view": len(
            missing_selected_view
        ),
    },
    "paths": {
        "mapping": str(mapping_path),
        "class_metadata": str(class_metadata_path),
        "ground_truth": str(GT_DIR),
        "features_placeholder": str(FEATURE_DIR),
        "splits": str(SPLIT_OUT_DIR),
        "segment_manifest": str(segment_manifest_path),
        "sequence_manifest": str(sequence_manifest_path),
        "validation_report": str(validation_report_path),
        "recording_download_manifest": str(
            recording_download_path
        ),
        "procedurevrl_extraction_manifest": str(
            procedurevrl_manifest_path
        ),
    },
    "next_step": (
        "Download the required v1 raw recordings, validate them with FFprobe, "
        "and crop/extract ProcedureVRL and CLIP-like visual features for each "
        "assembly/disassembly sequence."
    ),
}

summary_path = OUT_ROOT / "dataset_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))
print("\nSaved:", summary_path)

print("\nNext notebook:")
print("14_assembly101_v1_download_and_smoke_video_crops_COLAB.ipynb")

{
  "status": "completed",
  "dataset": "Assembly101",
  "annotation_granularity": "coarse",
  "task": "temporal action segmentation",
  "annotation_fps": 30.0,
  "raw_video_fps_description": "raw recordings are nominally 60 fps",
  "end_frame_convention": "exclusive",
  "selected_view": "v1",
  "selected_camera_file": "C10095_rgb.mp4",
  "output_root": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format",
  "num_official_action_classes": 202,
  "background_class_included": false,
  "num_model_classes": 202,
  "num_sequences": 680,
  "num_segments": 8753,
  "num_unique_recordings_required": 350,
  "num_recordings_already_downloaded": 0,
  "split_counts": {
    "train": 393,
    "val": 120,
    "test": 167
  },
  "activity_counts": {
    "assembly": 334,
    "disassembly": 346
  },
  "total_duration_hours": 40.20752777777778,
  "total_dense_frames_30fps": 4342413,
  "total_background_frames": 0,
  "total_conflicting_overlap_frames": 0,
  "validatio

## Expected successful result

The final cell should report:

```text
status: completed
annotation_granularity: coarse
num_official_action_classes: 202
num_sequences: > 0
num_segments: > 0
num_unique_recordings_required: <= 362
missing_ground_truth_files: 0
ground_truth_length_mismatches: 0
unknown_ground_truth_labels: 0
train/val/test overlap: 0
split_sequences_missing_annotations: 0
split_sequences_missing_selected_view: 0
```

The number of model classes may be:

```text
202
```

when the annotation crops are fully covered by action segments, or:

```text
203
```

when temporal gaps exist and the generated MS-TCN ground truth therefore includes a `background` class.